# Generating Text with a GPT Model

By the end of this notebook, you will understand the magic behind how a trained GPT model writes new text, one word at a time. You'll learn to control the "creativity" of its output using sampling techniques like temperature and top-k filtering. Finally, you will implement these core generation strategies yourself to bring a trained model to life.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Set a seed for reproducibility of random operations
torch.manual_seed(42)
np.random.seed(42)

# A simplified, "fake" GPT model for demonstration purposes.
# In a real scenario, this would be our trained Transformer model.
class FakeGPT(nn.Module):
    def __init__(self, vocab_size=50, block_size=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        # We don't need real weights, just a placeholder method.
    
    def forward(self, idx):
        # This fake model doesn't learn. It just returns random logits.
        # This is enough to demonstrate the *generation logic*.
        batch, seq_len = idx.shape
        # The output should be logits: raw scores for each token in the vocabulary.
        # Shape: (batch, sequence_length, vocab_size)
        random_logits = torch.randn(batch, seq_len, self.vocab_size)
        return random_logits, None # Return None for loss, as we aren't training

print("Setup complete. A FakeGPT class is defined for demonstration.")

### The Core Idea: Autoregressive Generation

How does a model write a story? It doesn't write the whole thing at once. It writes it one word (or, more accurately, one "token") at a time, in a process called **autoregressive generation**.

Think of it like the autocomplete on your phone, but on steroids.
1. You type: "I'm heading to the..."
2. Your phone suggests: "store", "gym", "movies". It has calculated the probabilities for the next word based on the context you provided.
3. You pick "store". Your new sentence is: "I'm heading to the store".
4. Now, the process repeats. The model uses this *new, longer sentence* as context to predict the *next* word.

This is a feedback loop: the model's output becomes its next input.



This simple, iterative process is the engine that drives all text generation in GPT models. The model only ever does one thing: predict the very next token. By stringing these predictions together, it can compose paragraphs, poems, or code.

In [ ]:
def generate_simple(model, start_tokens, max_new_tokens):
    """
    Generates text using a simple autoregressive loop.
    
    Args:
        model: A GPT-like model with a forward pass.
        start_tokens: A tensor of shape (batch, sequence_length) with starting token IDs.
        max_new_tokens: The number of new tokens to generate.
    
    Returns:
        The generated sequence of token IDs.
    """
    # Make a copy to avoid modifying the original
    generated_tokens = start_tokens

    for _ in range(max_new_tokens):
        # Get the model's predictions (logits) for the next token
        logits, _ = model(generated_tokens)
        
        # We only care about the prediction for the very last token in the sequence
        last_token_logits = logits[:, -1, :]
        
        # Convert logits to probabilities using the softmax function
        probabilities = F.softmax(last_token_logits, dim=-1)
        
        # Sample the next token from the probability distribution
        # This is like rolling a weighted die
        next_token = torch.multinomial(probabilities, num_samples=1)
        
        # Append the sampled token to our sequence
        generated_tokens = torch.cat((generated_tokens, next_token), dim=1)
        
    return generated_tokens

print("`generate_simple` function is defined.")

### Walking Through the Code

Let's break down our `generate_simple` function. It's a direct translation of the autoregressive idea.

- **`for _ in range(max_new_tokens):`**
  This is our main loop. We've told the model how many new tokens we want, and it will repeat the prediction process that many times.

- **`logits, _ = model(generated_tokens)`**
  Here, we feed the entire sequence of tokens generated *so far* into the model. The model returns `logits`, which are raw, unnormalized scores for every single token in our vocabulary at every position in the sequence.

- **`last_token_logits = logits[:, -1, :]`**
  We don't need predictions for tokens we already have. We only want to know what comes *next*. So, we slice the `logits` tensor to grab the scores corresponding to the very last token of our input sequence (`-1`).

- **`probabilities = F.softmax(last_token_logits, dim=-1)`**
  Logits are hard to work with. Softmax is a mathematical function that converts a vector of raw scores into a clean probability distribution, where all values are between 0 and 1 and sum to 1. Now, `probabilities[0][i]` tells us the model's confidence that token `i` is the correct next token.

- **`next_token = torch.multinomial(probabilities, num_samples=1)`**
  This is the sampling step. Instead of greedily picking the token with the absolute highest probability (which leads to boring, repetitive text), we perform a random sample from the distribution. `torch.multinomial` acts like rolling a heavily weighted die—tokens with higher probabilities are more likely to be chosen, but there's still a chance for less likely tokens to be picked, adding variety.

- **`generated_tokens = torch.cat((generated_tokens, next_token), dim=1)`**
  The feedback loop closes here. We append our newly chosen `next_token` to the end of our `generated_tokens` sequence. In the next iteration of the loop, this slightly longer sequence will become the new input to the model.

In [ ]:
# --- Demonstration ---
# Let's see it in action.

# 1. Initialize our fake model
VOCAB_SIZE = 50
model = FakeGPT(vocab_size=VOCAB_SIZE)

# 2. Create a starting sequence of tokens.
# This represents our prompt, e.g., "Hello, my name is"
# Shape: (batch_size, sequence_length) -> Here, one sentence with 5 tokens.
start_tokens = torch.tensor([[5, 12, 1, 8, 3]]) 
print(f"Starting sequence: {start_tokens}")
print(f"Starting sequence length: {start_tokens.shape[1]}")

# 3. Generate new tokens
NUM_NEW_TOKENS = 10
generated_sequence = generate_simple(
    model=model,
    start_tokens=start_tokens,
    max_new_tokens=NUM_NEW_TOKENS
)

print(f"\nGenerated sequence: {generated_sequence}")
print(f"Generated sequence length: {generated_sequence.shape[1]}")
print(f"\nThe first 5 tokens match the start_tokens, and {NUM_NEW_TOKENS} new ones were added.")

### Going Deeper: Controlling Creativity with Sampling

Our simple generation works, but what if the output is too random? Or too boring? We need knobs to control the model's behavior. This is where sampling strategies come in. The two most common are **Temperature** and **Top-K Filtering**.

#### Temperature
**Analogy:** Think of temperature as a "creativity" or "chaos" knob.

It modifies the probability distribution *before* we sample from it. We do this by dividing the logits by a temperature value: `logits / temperature`.

- **Low Temperature (e.g., 0.2):** This makes the distribution "sharper" or "spikier." The model becomes more confident and conservative, overwhelmingly favoring the most likely tokens. The output is more predictable and focused but can become repetitive.
- **Normal Temperature (1.0):** This is the default, with no effect on the logits.
- **High Temperature (e.g., 1.5):** This makes the distribution "flatter." The probabilities of likely and unlikely tokens become closer, increasing the chance of the model picking a surprising or creative word. This can lead to more interesting text, but also increases the risk of nonsense or grammatical errors.

#### Top-K Filtering
**Analogy:** Think of Top-K as creating a "shortlist of good candidates."

Instead of considering all 50,000+ tokens in the vocabulary, most of which are highly improbable, we simply restrict the model's choices to the `k` most likely tokens.

For example, with `top_k=50`, we first find the 50 tokens with the highest logits. Then, we set the logits of all other tokens to negative infinity, effectively removing them from the equation before we apply softmax and sample. This is a great way to prevent the model from picking truly bizarre words that might accidentally get a non-zero probability, especially when using high temperatures.

Let's implement a new `generate` function that includes these two powerful controls.

In [ ]:
def generate_with_sampling(model, start_tokens, max_new_tokens, temperature=1.0, top_k=None):
    """
    Generates text with temperature and top-k sampling.
    """
    generated_tokens = start_tokens
    
    for _ in range(max_new_tokens):
        # For simplicity, we'll crop the context if it gets too long
        # This is a detail from nanoGPT to handle fixed block sizes
        current_context = generated_tokens if generated_tokens.size(1) <= model.block_size else generated_tokens[:, -model.block_size:]
        
        # Forward pass
        logits, _ = model(current_context)
        
        # Pluck the logits for the final step
        last_token_logits = logits[:, -1, :]
        
        # 1. Apply temperature
        last_token_logits = last_token_logits / temperature
        
        # 2. Optionally apply top-k filtering
        if top_k is not None:
            # Find the top k values and their indices
            top_k_values, _ = torch.topk(last_token_logits, k=top_k)
            # The k-th value is our cutoff threshold
            cutoff_value = top_k_values[:, [-1]]
            # Set all logits less than the cutoff to -infinity
            last_token_logits[last_token_logits < cutoff_value] = -float('Inf')
            
        # Apply softmax to get probabilities
        probabilities = F.softmax(last_token_logits, dim=-1)
        
        # Sample the next token
        next_token = torch.multinomial(probabilities, num_samples=1)
        
        # Append and continue
        generated_tokens = torch.cat((generated_tokens, next_token), dim=1)
        
    return generated_tokens

# --- Visualization of Sampling Strategies ---

# Create some sample logits for a small vocabulary
vocab_size_vis = 15
logits_vis = torch.randn(1, vocab_size_vis) * 2 # Multiply to spread them out
# Make a few logits particularly high to simulate a real distribution
logits_vis[0, 3] = 6.0
logits_vis[0, 8] = 5.5
logits_vis[0, 2] = 4.0

# Calculate probabilities for different settings
probs_temp1 = F.softmax(logits_vis, dim=-1).squeeze().numpy()
probs_temp0_5 = F.softmax(logits_vis / 0.5, dim=-1).squeeze().numpy()
probs_temp1_5 = F.softmax(logits_vis / 1.5, dim=-1).squeeze().numpy()

# Calculate top-k probabilities
top_k_val = 5
logits_top_k = logits_vis.clone()
top_k_values, _ = torch.topk(logits_top_k, k=top_k_val)
cutoff = top_k_values[:, [-1]]
logits_top_k[logits_top_k < cutoff] = -float('Inf')
probs_top_k = F.softmax(logits_top_k, dim=-1).squeeze().numpy()


# Plotting
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("How Sampling Strategies Reshape Probability Distributions", fontsize=16)
token_indices = np.arange(vocab_size_vis)

# Temperature = 1.0 (Normal)
axs[0, 0].bar(token_indices, probs_temp1, color='skyblue')
axs[0, 0].set_title("Temperature = 1.0 (Normal)")
axs[0, 0].set_ylabel("Probability")

# Temperature = 0.5 (Conservative)
axs[0, 1].bar(token_indices, probs_temp0_5, color='cornflowerblue')
axs[0, 1].set_title("Temperature = 0.5 (More Confident/Spiky)")

# Temperature = 1.5 (Creative)
axs[1, 0].bar(token_indices, probs_temp1_5, color='lightsteelblue')
axs[1, 0].set_title("Temperature = 1.5 (More Creative/Flat)")
axs[1, 0].set_xlabel("Token ID")
axs[1, 0].set_ylabel("Probability")

# Top-K = 5
axs[1, 1].bar(token_indices, probs_top_k, color='slateblue')
axs[1, 1].set_title(f"Top-K = {top_k_val} (Restricted Choice)")
axs[1, 1].set_xlabel("Token ID")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Summary

**What We Built**
We constructed a complete text generation loop from scratch. We started with the core autoregressive engine and then layered on sophisticated sampling controls—`temperature` and `top_k`—to steer the model's output between conservative and creative. The visualization clearly shows how these parameters warp the underlying probability distribution, giving us fine-grained control over the final text.

**Key Takeaways**
- **Autoregressive Process:** Text generation is fundamentally iterative: predict the next token based on past tokens, append it, and repeat.
- **Sampling over Greed:** We *sample* from the model's predicted probabilities rather than always picking the most likely token. This is crucial for generating varied and interesting text.
- **Temperature is the Chaos Knob:** Lower temperatures lead to focused, repetitive text. Higher temperatures produce diverse, surprising, and sometimes nonsensical output.
- **Top-K is the Sanity Check:** It prevents the model from choosing bizarre, low-probability tokens by limiting the pool of candidates to the most plausible options.

**What's Next**
This notebook marks the end of our journey through nanoGPT. You now understand the entire lifecycle of a GPT model: from the fundamental Transformer `Block` architecture, to the `GPT` model that stacks them, through the `Training Loop` that teaches the model, and finally to this `Generator` that brings its knowledge to life. With these building blocks, you are equipped to explore, modify, and build upon foundational language models.